# B - MV-FraudGT head ablation - AML-Small-HI - seed 42
Experiment B keeps the same full FraudGT encoder and weighted cross-entropy as A, but replaces the original edge head with the multi-view gated head.

In [ ]:
import platform, sys, subprocess, torch
print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.2f} GiB')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
import subprocess, sys, torch
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_url)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'torchmetrics', 'yacs', 'datatable',
                'pandas', 'wandb', 'ogb', 'tensorboardX'], check=True)
print('Dependencies installed. Restart the session only if imports fail.')

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/mhiunguyen/MV-IA-FraudGT.git'
repo = Path('/kaggle/working/MV-IA-FraudGT')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
os.chdir(repo)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from pathlib import Path
from shutil import copy2
candidates = list(Path('/kaggle/input').rglob('HI-Small_Trans.csv'))
if not candidates:
    raise FileNotFoundError('Add the IBM AML dataset as Kaggle Input; HI-Small_Trans.csv was not found.')
destination = Path('/kaggle/working/MV-IA-FraudGT/data/AML/HI-Small_Trans.csv')
destination.parent.mkdir(parents=True, exist_ok=True)
if not destination.exists() or destination.stat().st_size != candidates[0].stat().st_size:
    copy2(candidates[0], destination)
print('Dataset:', destination, f'({destination.stat().st_size / 1024**2:.1f} MiB)')

## One-seed training
The first run may spend roughly 10–20 minutes generating Ports on CPU before GPU training begins. The T4 configuration uses batch 256, fanout `[15,15]`, 128 iterations/epoch and 50 epochs.

In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working/MV-IA-FraudGT')
cmd = [sys.executable, '-u', '-m', 'fraudGT.main',
       '--cfg', 'configs/AML-Small-HI/AML-Small-HI-MV-FraudGT-T4.yaml',
       '--repeat', '1', '--gpu', '0', 'name_tag', 'B-MV-FraudGT-1Seed']
print('Command:', ' '.join(cmd))
started = time.time()
subprocess.run(cmd, check=True)
print(f'Elapsed: {(time.time() - started) / 60:.1f} minutes')

In [ ]:
from pathlib import Path
run_dir = Path('results/AML-Small-HI-MV-FraudGT-T4-B-MV-FraudGT-1Seed-gpu0')
summary = Path('/kaggle/working/summary_B_MV_FraudGT_seed42.csv')
subprocess.run([sys.executable, 'scripts/summarize_thresholds.py', str(run_dir),
                '--output', str(summary), '--fixed-threshold', '0.10'], check=True)
print('Save this CSV and capture the summary table:', summary)